# Example 06 — Multi-DOF, Multi-Nonlinearity FRF

FRF for all DOFs of a 4-DOF chain with two different nonlinear elements (cubic spring at DOF 0 and tanh dry friction at DOF 2), plus a continuation step-size history plot as a convergence proxy. (Krack & Gross 2019)

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.systems.oscillators import ChainOfOscillators
from nlvib.nonlinearities.elements import cubic_spring, tanh_dry_friction
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions

In [ ]:
# --- System setup ---
MASSES = [1.0, 1.5, 1.0, 0.8]
STIFFNESSES = [1.0, 0.8, 1.2, 0.6, 1.0]
DAMPINGS = [0.02, 0.02, 0.02, 0.02, 0.02]
FORCE_AMP = 0.1
OMEGA_MIN = 0.3
OMEGA_MAX = 1.5
N_HARMONICS = 3

system = ChainOfOscillators(masses=MASSES, stiffnesses=STIFFNESSES, dampings=DAMPINGS)
system.add_nonlinear_element(cubic_spring(k3=0.5, dof_index=0))
system.add_nonlinear_element(tanh_dry_friction(f0=0.2, c=8.0, dof_index=2))
excitation = {'dof': 0, 'amplitude': FORCE_AMP, 'harmonic': 1}
n_dof = system.n_dof
print(f'System: n_dof={n_dof}')

In [ ]:
# --- Initial solution + continuation ---
n_total = n_dof * (2 * N_HARMONICS + 1)
Q0 = np.zeros(n_total)
for _ in range(40):
    R, J = hb_residual(Q0, OMEGA_MIN, system, N_HARMONICS, excitation)
    if np.linalg.norm(R) < 1e-10: break
    try: dQ = np.linalg.solve(J, -R)
    except np.linalg.LinAlgError: dQ = np.linalg.lstsq(J, -R, rcond=None)[0]
    Q0 += dQ

def residual_fn(Q, omega):
    return hb_residual(Q, omega, system, N_HARMONICS, excitation)

opts = ContinuationOptions(
    ds_initial=0.01, ds_min=1e-6, ds_max=0.05, max_steps=1000,
    newton_tol=1e-8, lambda_min=OMEGA_MIN, lambda_max=OMEGA_MAX,
)
result = ContinuationSolver().run(residual_fn, Q0, OMEGA_MIN, opts)
print(f'Continuation: {result.n_steps} steps, {result.message}')

In [ ]:
# --- Extract amplitudes and save plots ---
solutions = result.solutions
omegas = solutions[:, -1]
stability = result.stability

amplitudes = np.zeros((n_dof, len(omegas)))
for dof in range(n_dof):
    cos1 = solutions[:, n_dof*1 + dof]
    sin1 = solutions[:, n_dof*2 + dof]
    amplitudes[dof, :] = np.sqrt(cos1**2 + sin1**2)

output_dir = Path('..') / 'examples' / '06_multi_dof_multi_nl' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

fig_frf, axes_frf = plt.subplots(n_dof, 1, figsize=(8, 10), sharex=True)
for dof in range(n_dof):
    ax = axes_frf[dof]; amp = amplitudes[dof, :]
    for i in range(len(omegas) - 1):
        is_stable = not bool(stability[i])
        ax.plot(omegas[i:i+2], amp[i:i+2],
                color='tab:blue' if is_stable else 'tab:red',
                linestyle='-' if is_stable else '--', linewidth=1.2)
    ax.set_ylabel(f'$|q_{dof}|$'); ax.set_title(f'DOF {dof}')
handles = [
    Line2D([0], [0], color='tab:blue', linestyle='-', label='stable'),
    Line2D([0], [0], color='tab:red', linestyle='--', label='unstable'),
]
axes_frf[0].legend(handles=handles, loc='upper left')
axes_frf[-1].set_xlabel(r'Excitation frequency $\Omega$ (rad/s)')
fig_frf.suptitle('Example 06 — Multi-DOF Multi-NL FRF (all DOFs)')
fig_frf.tight_layout()
fig_frf.savefig(output_dir / 'frf_all_dofs.png', dpi=150)
print('Saved frf_all_dofs.png')

fig_conv, ax_conv = plt.subplots(figsize=(8, 4))
ds_history = result.ds_history[1:]
ax_conv.semilogy(np.arange(1, len(ds_history)+1), ds_history, color='tab:green', linewidth=1.2)
ax_conv.set_xlabel('Continuation step'); ax_conv.set_ylabel('Arc-length step size ds')
ax_conv.set_title('Example 06 — Newton Convergence Proxy (ds history)')
ax_conv.grid(True, which='both', alpha=0.4); fig_conv.tight_layout()
fig_conv.savefig(output_dir / 'convergence.png', dpi=150)
print('Saved convergence.png')

In [ ]:
# --- Display FRF all DOFs inline ---
fig_frf

In [ ]:
# --- Display convergence proxy inline ---
fig_conv

In [ ]:
# --- Summary ---
print('=' * 55)
print('SUMMARY — Example 06: Multi-DOF Multi-NL Peak Amplitudes')
print('=' * 55)
for dof in range(n_dof):
    amp = amplitudes[dof, :]
    if len(amp) > 0:
        peak_idx = int(np.argmax(amp))
        print(f'  DOF {dof}: peak amplitude = {float(amp[peak_idx]):.6f}  at omega = {float(omegas[peak_idx]):.4f} rad/s')
    else:
        print(f'  DOF {dof}: no data')
print('=' * 55)